# Solar filament segmentation — U-Net baseline

Pipeline: grayscale H-alpha (2048²) → downscale to 1024 → U-Net(ResNet34) filament probability → 8-way TTA → bicubic upsample → threshold → connected components = one instance per filament → COCO RLE.

Training target = mean of the annotators' masks per image, kept soft (area fraction) when downscaled. Post-processing (threshold / min area / gap-linking) is tuned on a 15 % hold-out split grouped by observation day, with a local Panoptic Quality implementation that matches the host's self-evaluation notebook.

**Settings needed:** Accelerator = GPU T4 x2 or P100, Internet = On (phone-verified account).  
Full run ≈ 2 h on T4. Outputs land in `/kaggle/working/outputs/`: `submission.csv`, `best.pt` (+ `.json`), `postprocess.json`, `report.json`, `environment.json`.

In [ ]:
MODE = 'train'   # 'train' = full pipeline; 'predict' = reproduce from saved weights (see last section)
WEIGHTS_DIR = '/kaggle/input/segmenthreads-weights'   # only used when MODE == 'predict'

In [ ]:
!pip install -q segmentation-models-pytorch==0.6.0 pycocotools==2.0.11
import torch, sys
assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU, then restart the session'
print(sys.version.split()[0], torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
import os
os.makedirs('filament', exist_ok=True)

In [ ]:
%%writefile filament/__init__.py
"""Solar filament instance segmentation — baseline pipeline (Kaggle filament-segmentation-2026)."""


In [ ]:
%%writefile filament/data.py
"""Loading MAGFiLO COCO annotations and rasterising them into masks.

Key facts (verified on the 2026 Kaggle release):
- 707 train JPEGs, 2048x2048, 8-bit grayscale.
- 1154 COCO "image" entries: one per (annotator set, file). A file can have up to 3 sets.
- Each annotation is a single polygon; rasterisation uses pycocotools, the same library the
  host uses for RLE, so our masks match the official ground-truth convention.
"""
from __future__ import annotations

import json
import random
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import numpy as np
from pycocotools import mask as mask_utils

IMG_SIZE = 2048
ANN_JSON = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"


@dataclass
class Sample:
    """One annotator's annotation set for one image file."""
    set_id: str                    # e.g. "010401-20160920230134Lh"
    file_name: str                 # e.g. "20160920230134Lh.jpeg"
    polygons: list[list[float]] = field(default_factory=list)  # each [x0,y0,x1,y1,...]

    @property
    def stem(self) -> str:
        return Path(self.file_name).stem


def find_dataset_root(start: str | Path) -> Path:
    """Return the MAGFiLO_1.0_Kaggle_2026 folder under `start` (works locally and on Kaggle)."""
    start = Path(start)
    direct = start / "filament-segmentation-2026" / "MAGFiLO_1.0_Kaggle_2026" / "train" / ANN_JSON
    hits = [direct] if direct.exists() else list(start.rglob(ANN_JSON))
    if not hits:
        raise FileNotFoundError(f"annotation json not found under {start}")
    return hits[0].parent.parent  # .../MAGFiLO_1.0_Kaggle_2026


def train_image_dir(root: Path) -> Path:
    return root / "train" / "train_images"


def test_image_dir(root: Path) -> Path:
    return root / "test" / "test_images"


def load_samples(root: Path) -> list[Sample]:
    """One Sample per COCO image entry (= one annotator's set for one file), polygons attached.

    Every annotation in the 2026 release is a single polygon (checked: 8199/8199, no
    iscrowd/RLE, none with < 3 points); the check below makes a future format change loud
    instead of silently dropping pieces.
    """
    with (root / "train" / ANN_JSON).open(encoding="utf-8") as fh:
        coco = json.load(fh)
    by_set: dict[str, Sample] = {
        im["id"]: Sample(set_id=im["id"], file_name=im["file_name"]) for im in coco["images"]
    }
    for a in coco["annotations"]:
        seg = a["segmentation"]
        if a.get("iscrowd", 0) or not isinstance(seg, list) or len(seg) != 1 or len(seg[0]) < 6:
            raise ValueError(f"unexpected segmentation format in annotation {a['id']}")
        by_set[a["image_id"]].polygons.append(seg[0])
    return list(by_set.values())


def split_by_file(samples: list[Sample], val_frac: float = 0.15,
                  seed: int = 42) -> tuple[list[Sample], list[Sample]]:
    """Hold out ~val_frac of the *files*, grouped by observation date (YYYYMMDD).

    All annotator sets of a file land on the same side, and so do files taken the same
    day from different GONG sites (two val files used to have a same-minute twin in train
    with image correlation 0.97, i.e. a leaked duplicate).
    """
    files = sorted({s.file_name for s in samples})
    days = sorted({f[:8] for f in files})
    rng = random.Random(seed)
    rng.shuffle(days)
    target = int(round(len(files) * val_frac))
    val_files: set[str] = set()
    for d in days:
        if len(val_files) >= target:
            break
        val_files |= {f for f in files if f[:8] == d}
    train = [s for s in samples if s.file_name not in val_files]
    val = [s for s in samples if s.file_name in val_files]
    return train, val


def polygon_to_mask(poly: list[float], h: int = IMG_SIZE, w: int = IMG_SIZE) -> np.ndarray:
    """Rasterise one COCO polygon with pycocotools (matches the host's `area` field exactly)."""
    rles = mask_utils.frPyObjects([poly], h, w)
    return mask_utils.decode(mask_utils.merge(rles)).astype(np.uint8)


def instance_masks(sample: Sample) -> list[np.ndarray]:
    """One uint8 2048x2048 mask per filament in the annotation set."""
    return [polygon_to_mask(p) for p in sample.polygons]


def semantic_mask(sample: Sample) -> np.ndarray:
    """Union of all filaments in one annotation set, uint8 {0,1}, 2048x2048."""
    m = np.zeros((IMG_SIZE, IMG_SIZE), np.uint8)
    for p in sample.polygons:
        m |= polygon_to_mask(p)
    return m


def read_gray(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f"cannot read {path}")
    return img


In [ ]:
%%writefile filament/rle.py
"""Submission encoding: COCO compressed RLE *counts* string for a 2048x2048 binary mask.

The host decodes with pycocotools using size [2048, 2048]; we encode with the same library
in Fortran order, so a mask survives the round trip exactly (checked in tests).
"""
import numpy as np
from pycocotools import mask as mask_utils

from .data import IMG_SIZE

SIZE = (IMG_SIZE, IMG_SIZE)


def encode_counts(mask: np.ndarray) -> str:
    """Binary HxW mask -> COCO compressed RLE counts (ASCII 48..111; may contain '\')."""
    if mask.shape != SIZE:
        raise ValueError(f"mask must be {SIZE}, got {mask.shape}")
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    return rle["counts"].decode("ascii")


def decode_counts(counts: str) -> np.ndarray:
    """Inverse of encode_counts: counts string -> uint8 HxW mask (the host's decode path)."""
    return mask_utils.decode({"size": list(SIZE), "counts": counts.encode("ascii")})


In [ ]:
%%writefile filament/metric.py
"""Local re-implementation of Panoptic Quality (Kirillov et al., CVPR 2019) for one class.

PQ = sum_{TP} IoU / (|TP| + 0.5|FP| + 0.5|FN|), a pair is TP when IoU > 0.5
(IoU > 0.5 guarantees the matching is unique, as proven in the PQ paper).

Checked against the host's public self-evaluation notebook (Kaggle, azimahmadzadeh): same
strict IoU > 0.5 rule, same FP/FN counting, same dataset-level aggregation; identical
TP/FP/FN/PQ on real annotator-vs-annotator data. The host additionally applies unspecified
"verification strategies" for edge cases, so treat the number as a very close estimate.
"""
from __future__ import annotations

import numpy as np


def _bbox(m: np.ndarray):
    ys, xs = np.nonzero(m)
    if len(ys) == 0:
        return None
    return ys.min(), ys.max() + 1, xs.min(), xs.max() + 1


def _iou_matrix(gts: list[np.ndarray], preds: list[np.ndarray]) -> np.ndarray:
    """IoU for every (gt, pred) pair; only pairs whose bounding boxes overlap are computed."""
    iou = np.zeros((len(gts), len(preds)))
    gb = [_bbox(m) for m in gts]
    pb = [_bbox(m) for m in preds]
    g_area = [int(m.sum()) for m in gts]
    p_area = [int(m.sum()) for m in preds]
    for i, b1 in enumerate(gb):
        if b1 is None:
            continue
        for j, b2 in enumerate(pb):
            if b2 is None:
                continue
            y0, y1 = max(b1[0], b2[0]), min(b1[1], b2[1])
            x0, x1 = max(b1[2], b2[2]), min(b1[3], b2[3])
            if y0 >= y1 or x0 >= x1:
                continue
            inter = int(np.logical_and(gts[i][y0:y1, x0:x1], preds[j][y0:y1, x0:x1]).sum())
            if inter:
                iou[i, j] = inter / (g_area[i] + p_area[j] - inter)
    return iou


def pq_stats(gts: list[np.ndarray], preds: list[np.ndarray]) -> dict:
    """Return TP/FP/FN counts and summed TP IoU for one image."""
    iou = _iou_matrix(gts, preds)
    tp_mask = iou > 0.5
    tp = int(tp_mask.sum())
    return {
        "tp": tp,
        "fp": len(preds) - int(tp_mask.any(0).sum()),
        "fn": len(gts) - int(tp_mask.any(1).sum()),
        "iou_sum": float(iou[tp_mask].sum()),
    }


def gt_crops(gts: list[np.ndarray]) -> list[tuple]:
    """Precompute (y0, y1, x0, x1, bool crop, area) per GT mask; reuse across many evaluations."""
    out = []
    for m in gts:
        b = _bbox(m)
        if b is None:  # empty GT mask: keep it so it still counts as a FN (host behaviour)
            out.append((0, 0, 0, 0, np.zeros((0, 0), bool), 0))
            continue
        y0, y1, x0, x1 = b
        crop = m[y0:y1, x0:x1].astype(bool)
        out.append((y0, y1, x0, x1, crop, int(crop.sum())))
    return out


def pq_stats_labels(gt: list[tuple], labels: np.ndarray) -> dict:
    """Fast PQ stats when predictions are a label map (0 = background, 1..K = disjoint instances).

    Gives the same result as pq_stats() for disjoint predictions (checked in tests).
    """
    k = int(labels.max())
    p_area = np.bincount(labels.ravel(), minlength=k + 1)
    matched_p = set(); tp = 0; iou_sum = 0.0
    for y0, y1, x0, x1, crop, g_area in gt:
        if g_area == 0:
            continue
        inter = np.bincount(labels[y0:y1, x0:x1][crop], minlength=k + 1)
        inter[0] = 0
        j = int(inter.argmax())          # IoU > 0.5 can only happen for the largest overlap
        if inter[j] == 0:
            continue
        iou = inter[j] / (g_area + p_area[j] - inter[j])
        if iou > 0.5:
            tp += 1; iou_sum += float(iou); matched_p.add(j)
    return {"tp": tp, "fp": k - len(matched_p), "fn": len(gt) - tp, "iou_sum": iou_sum}


def pq_from_stats(stats: list[dict]) -> dict:
    """Aggregate over the dataset (the leaderboard definition) plus diagnostics.

    `PQ` is what the host computes. `PQ_per_image_mean`, `SQ`, `RQ` are diagnostics only
    (an image with no GT and no prediction counts as 1.0 in the per-image mean).
    """
    tp = sum(s["tp"] for s in stats); fp = sum(s["fp"] for s in stats); fn = sum(s["fn"] for s in stats)
    iou_sum = sum(s["iou_sum"] for s in stats)
    denom = tp + 0.5 * fp + 0.5 * fn
    per_img = [s["iou_sum"] / d if (d := s["tp"] + 0.5 * s["fp"] + 0.5 * s["fn"]) > 0 else 1.0 for s in stats]
    return {
        "PQ": iou_sum / denom if denom else 0.0,
        "PQ_per_image_mean": float(np.mean(per_img)) if per_img else 0.0,
        "SQ": iou_sum / tp if tp else 0.0,
        "RQ": tp / denom if denom else 0.0,
        "TP": tp, "FP": fp, "FN": fn,
    }


In [ ]:
%%writefile filament/model.py
"""U-Net (segmentation_models_pytorch) predicting a 1-channel 'filament' probability map,
the training loss, and weight save/load helpers."""
from __future__ import annotations

import json
from pathlib import Path

import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import torch.nn.functional as F


def build_model(encoder: str = "resnet34", pretrained: bool = True) -> nn.Module:
    return smp.Unet(
        encoder_name=encoder,
        encoder_weights="imagenet" if pretrained else None,
        in_channels=1,      # images are grayscale; smp sums the RGB weights of the first conv
        classes=1,
    )


class BceDiceLoss(nn.Module):
    """BCE (accepts soft targets) + soft Dice against the heavy class imbalance (~0.4 % foreground).

    The logits are cast to fp32 first so the loss is safe even if called outside autocast
    (a 1024x1024 sum overflows fp16).
    """

    def __init__(self, bce_weight: float = 0.5):
        super().__init__()
        self.w = bce_weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        logits = logits.float()
        bce = F.binary_cross_entropy_with_logits(logits, target)
        p = torch.sigmoid(logits)
        inter = (p * target).sum((1, 2, 3))
        dice = 1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + 1)
        return self.w * bce + (1 - self.w) * dice.mean()


def save_weights(model: nn.Module, path: Path, encoder: str, size: int, half: bool = True) -> None:
    """Save the state_dict (fp16 by default: 47 MB instead of 93 MB, fits in a git repo)
    plus a sidecar JSON with what is needed to rebuild the network."""
    sd = model.state_dict()
    if half:
        sd = {k: (v.half() if v.is_floating_point() else v) for k, v in sd.items()}
    torch.save(sd, path)
    with Path(str(path) + ".json").open("w", encoding="utf-8") as fh:
        json.dump({"encoder": encoder, "size": size, "in_channels": 1, "classes": 1, "half": half}, fh)


def load_weights(path: Path, device: str) -> tuple[nn.Module, dict]:
    """Rebuild the model from the sidecar JSON and load the (possibly fp16) state_dict."""
    with Path(str(path) + ".json").open(encoding="utf-8") as fh:
        meta = json.load(fh)
    model = build_model(meta["encoder"], pretrained=False)
    sd = torch.load(path, map_location="cpu")
    model.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in sd.items()})
    return model.to(device).eval(), meta


In [ ]:
%%writefile filament/dataset.py
"""PyTorch dataset: downscaled whole-disk images + soft filament targets, cached in RAM.

Design (measured on the 2026 release, see README "Design decisions"):
- One item per *file*, not per annotator set. Up to 3 annotators mark the same image and
  their union masks agree only moderately (median IoU 0.47), so the target is the pixelwise
  mean of the annotators' masks. Same optimum as feeding the sets separately, but with
  less gradient noise and without weighting multi-annotated files 2-3x.
- The 2048 mask is downscaled with INTER_AREA and kept *soft* (area fraction per pixel)
  instead of thresholded: thin filaments (widest point median 15.6 px, 5th pct 8 px at 2048) lose ~10 % IoU
  when binarised at 1024, but almost nothing when the fraction is kept and the prediction
  is upsampled bicubically (oracle PQ ceiling 0.90 -> 0.99).
"""
from __future__ import annotations

from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

from .data import Sample, read_gray, semantic_mask

# Fixed normalisation; the exact values do not matter much because the encoder starts
# with BatchNorm, but train and inference must use the same ones.
MEAN, STD = 0.5, 0.25

# Photometric jitter ranges: GONG sites differ in exposure/seeing. Disk mean varies 99-130
# across train+test (ratio ~1.3); filaments are ~12 % darker than their surroundings.
GAIN_RANGE = (0.80, 1.25)
OFFSET_RANGE = (-0.08, 0.08)
GAMMA_RANGE = (0.80, 1.25)


def normalize(img_u8: np.ndarray) -> torch.Tensor:
    """uint8 HxW -> float tensor 1xHxW with the training normalisation."""
    return torch.from_numpy(np.ascontiguousarray(img_u8)).float().div(255).sub(MEAN).div(STD)[None]


def to_tensor(img_small: np.ndarray) -> torch.Tensor:
    """uint8 HxW -> 1x1xHxW tensor, for inference."""
    return normalize(img_small)[None]


def soft_target(samples_of_file: list[Sample], size: int) -> np.ndarray:
    """Mean over annotator sets of the INTER_AREA-downscaled filament mask, as uint8 (x255)."""
    acc = np.zeros((size, size), np.float32)
    for s in samples_of_file:
        m = semantic_mask(s).astype(np.float32)
        acc += cv2.resize(m, (size, size), interpolation=cv2.INTER_AREA)
    acc /= len(samples_of_file)
    return np.rint(acc * 255).astype(np.uint8)


class FilamentDataset(Dataset):
    """Items: (normalised image 1xSxS, soft target 1xSxS in [0,1]). One item per image file."""

    def __init__(self, samples: list[Sample], image_dir: Path, size: int = 1024, augment: bool = False):
        self.size, self.augment = size, augment
        by_file: dict[str, list[Sample]] = defaultdict(list)
        for s in samples:
            by_file[s.file_name].append(s)
        self.files = sorted(by_file)
        self.items: list[tuple[np.ndarray, np.ndarray]] = []
        for f in self.files:
            img = cv2.resize(read_gray(image_dir / f), (size, size), interpolation=cv2.INTER_AREA)
            self.items.append((img, soft_target(by_file[f], size)))

    def __len__(self) -> int:
        return len(self.items)

    def _augment(self, img: np.ndarray, m: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        k = np.random.randint(4)                      # the 8 symmetries of the square
        img, m = np.rot90(img, k), np.rot90(m, k)
        if np.random.rand() < 0.5:
            img, m = img[:, ::-1], m[:, ::-1]
        f = img.astype(np.float32) / 255.0
        gain = np.random.uniform(*GAIN_RANGE)
        offset = np.random.uniform(*OFFSET_RANGE)
        gamma = np.random.uniform(*GAMMA_RANGE)
        # keep the off-disk background exactly black, as it is in every real image
        f = np.where(f > 0, np.clip(f * gain + offset, 0, 1) ** gamma, 0.0)
        return (f * 255).astype(np.uint8), m

    def __getitem__(self, i: int) -> tuple[torch.Tensor, torch.Tensor]:
        img, m = self.items[i]
        if self.augment:
            img, m = self._augment(img, m)
        x = normalize(img)
        y = torch.from_numpy(np.ascontiguousarray(m)).float().div(255)[None]
        return x, y


In [ ]:
%%writefile filament/postprocess.py
"""Probability map -> instances (one per filament).

Instances are the connected components of the thresholded map. PQ punishes fragmentation,
so components may optionally be labelled on a slightly *dilated* mask (pieces up to
2*link_px apart join into one filament) and are then cut back to the undilated pixels.
Components smaller than `min_area` pixels are dropped.

Everything is done on a single int32 label map (0 = background, 1..K = instances), so
the cost is independent of the number of components. A noisy map with thousands of blobs
used to take seconds and gigabytes per call when every instance was a full 2048x2048 array.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass

import cv2
import numpy as np


@dataclass(frozen=True)
class PostprocessParams:
    """The three tuned post-processing knobs (chosen on the validation split by PQ)."""
    thr: float = 0.5        # probability threshold for 'filament' pixels
    min_area: int = 200     # drop components smaller than this (pixels at 2048x2048)
    link_px: int = 0        # dilation radius used only to decide which pieces belong together

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "PostprocessParams":
        return cls(thr=float(d["thr"]), min_area=int(d["min_area"]), link_px=int(d["link_px"]))


# Grid searched on the validation split. GT filament areas at 2048: 1st percentile 209 px,
# 5th 317, median 1228 (scripts/measure_filaments.py), so min_area above ~300 can only lose true
# filaments; the oracle experiment also showed link_px > 0 merges neighbours, so 0 is
# included and expected to win unless the model fragments filaments.
DEFAULT_GRID = {
    "thr": (0.3, 0.4, 0.5, 0.6),
    "min_area": (50, 100, 150, 200, 300),
    "link_px": (0, 3, 6),
}


def prob_to_labels(prob: np.ndarray, thr: float = 0.5, min_area: int = 200,
                   link_px: int = 0) -> np.ndarray:
    """prob: float32 HxW in [0,1] at full resolution. Returns an int32 label map."""
    fg = (prob >= thr).astype(np.uint8)
    lab_src = fg
    if link_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * link_px + 1, 2 * link_px + 1))
        lab_src = cv2.dilate(fg, k)
    n, labels = cv2.connectedComponents(lab_src, connectivity=8)
    labels = labels * fg                        # cut back to the undilated pixels
    area = np.bincount(labels.ravel(), minlength=n)
    keep = area >= max(int(min_area), 1)
    keep[0] = False
    lut = np.zeros(n, np.int32)
    lut[keep] = np.arange(1, int(keep.sum()) + 1, dtype=np.int32)
    return lut[labels]


def labels_to_instances(labels: np.ndarray):
    """Yield one uint8 HxW mask per instance (1..K), in label order."""
    for k in range(1, int(labels.max()) + 1):
        yield (labels == k).astype(np.uint8)


def prob_to_instances(prob: np.ndarray, thr: float = 0.5, min_area: int = 200,
                      link_px: int = 0) -> list[np.ndarray]:
    """Convenience wrapper: list of binary instance masks (same rule as prob_to_labels)."""
    return list(labels_to_instances(prob_to_labels(prob, thr, min_area, link_px)))


In [ ]:
%%writefile filament/inference.py
"""Model inference at reduced resolution with test-time augmentation, upsampled to 2048x2048."""
from __future__ import annotations

import cv2
import numpy as np
import torch
import torch.nn as nn

from .dataset import to_tensor


def _d4_transforms(x: torch.Tensor, full: bool):
    """Yield (transformed input, function that maps a prediction back).

    `full=False`: the 4 flip combinations. `full=True`: all 8 symmetries of the square
    (rotations + flips), matching the rot90/flip augmentation used in training.
    """
    rots = range(4) if full else (0,)
    for k in rots:
        r = torch.rot90(x, k, (2, 3)) if k else x
        yield r, (lambda p, k=k: torch.rot90(p, -k, (2, 3)) if k else p)
        yield torch.flip(r, [3]), (lambda p, k=k: torch.rot90(torch.flip(p, [3]), -k, (2, 3)) if k
                                   else torch.flip(p, [3]))
        if not full:  # the two remaining flips of the 4-flip set
            yield torch.flip(r, [2]), (lambda p: torch.flip(p, [2]))
            yield torch.flip(r, [2, 3]), (lambda p: torch.flip(p, [2, 3]))


@torch.no_grad()
def predict_prob(model: nn.Module, img_full: np.ndarray, size: int, device: str,
                 tta: bool = True, tta_full: bool = False) -> np.ndarray:
    """Return a float32 probability map at the input image's resolution (2048x2048).

    The network runs at `size` x `size`; the probability map is upsampled with bicubic
    interpolation, which preserves the sub-pixel information in the soft output
    (measured on GT round trips: PQ ceiling 0.991 bicubic vs 0.975 bilinear at 1024).
    """
    small = cv2.resize(img_full, (size, size), interpolation=cv2.INTER_AREA)
    x = to_tensor(small).to(device)
    acc, n = None, 0
    with torch.autocast(device_type="cuda", enabled=device.startswith("cuda")):
        for xi, back in (_d4_transforms(x, tta_full) if tta else [(x, lambda p: p)]):
            p = back(torch.sigmoid(model(xi).float()))
            acc = p if acc is None else acc + p
            n += 1
    prob = (acc / n)[0, 0].cpu().numpy().astype(np.float32)
    prob = cv2.resize(prob, (img_full.shape[1], img_full.shape[0]), interpolation=cv2.INTER_CUBIC)
    return np.clip(prob, 0.0, 1.0)


In [ ]:
%%writefile filament/submission.py
"""Write and verify the competition CSV (filament_id, segmentation_rle)."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from .rle import decode_counts, encode_counts

HEADER = "filament_id,segmentation_rle"


def rows_for_image(stem: str, instances) -> list[tuple[str, str]]:
    """`<stem>_<k>` ids starting at 1 (the format in the host's example), one row per instance."""
    return [(f"{stem}_{k}", encode_counts(m)) for k, m in enumerate(instances, 1)]


def write_submission(rows: list[tuple[str, str]], path: Path) -> None:
    """Write raw lines.

    COCO RLE counts use ASCII 48..111, which may include a backslash but never ',' or
    quotes, so no CSV quoting/escaping is needed (the host forbids quotes, and an
    escapechar would corrupt the backslashes).
    """
    for fid, counts in rows:
        if "," in counts or '"' in counts or "'" in counts or "_" not in fid:
            raise ValueError(f"unexpected characters in row {fid}")
    with open(path, "w", newline="") as fh:
        fh.write(HEADER + "\n")
        fh.writelines(f"{fid},{counts}\n" for fid, counts in rows)


def verify_submission(path: Path, rows: list[tuple[str, str]], expected_stems: set[str],
                      n_decode: int = 5) -> pd.DataFrame:
    """Read the file back the way the host does (pandas defaults) and check it matches."""
    sub = pd.read_csv(path)
    if list(sub.columns) != HEADER.split(","):
        raise RuntimeError(f"bad header {list(sub.columns)}")
    if len(sub) != len(rows) or sub["segmentation_rle"].tolist() != [r[1] for r in rows]:
        raise RuntimeError("CSV round trip changed the RLE strings")
    stems = sub["filament_id"].str.split("_", n=1).str[0]   # the host parses on the FIRST '_'
    if not stems.isin(expected_stems).all():
        raise RuntimeError("filament_id does not start with a known image id")
    if sub["filament_id"].duplicated().any():
        raise RuntimeError("duplicate filament_id")
    for counts, (_, orig) in zip(sub["segmentation_rle"].head(n_decode), rows[:n_decode]):
        if not np.array_equal(decode_counts(counts), decode_counts(orig)):
            raise RuntimeError("decoded mask differs after CSV round trip")
    return sub


In [ ]:
%%writefile train.py
"""Train the U-Net, tune post-processing on a held-out split by local PQ, write submission.csv.

Works locally and on Kaggle (paths are auto-detected).
  Kaggle GPU, full run:          python train.py
  Local CPU smoke test:          python train.py --smoke
  Reproduce from saved weights:  python train.py --predict-only --weights weights/unet_r34_1024.pt \
                                                 --postprocess weights/postprocess.json
"""
from __future__ import annotations

import argparse
import itertools
import json
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from filament.data import (find_dataset_root, instance_masks, load_samples, read_gray, split_by_file,
                           test_image_dir, train_image_dir)
from filament.dataset import FilamentDataset
from filament.inference import predict_prob
from filament.metric import gt_crops, pq_from_stats, pq_stats_labels
from filament.model import BceDiceLoss, build_model, load_weights, save_weights
from filament.postprocess import DEFAULT_GRID, PostprocessParams, labels_to_instances, prob_to_labels
from filament.submission import rows_for_image, verify_submission, write_submission

SELECT_THRS = (0.3, 0.4, 0.5, 0.6)   # thresholds tried when ranking checkpoints


def parse_args() -> argparse.Namespace:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", default=None, help="folder that contains MAGFiLO_1.0_Kaggle_2026")
    ap.add_argument("--out", default="outputs")
    ap.add_argument("--size", type=int, default=1024, help="network input resolution")
    ap.add_argument("--encoder", default="resnet34")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--bs", type=int, default=4)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--val-frac", type=float, default=0.15)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--no-pretrained", action="store_true")
    ap.add_argument("--no-tta8", action="store_true", help="use 4 flips instead of 8 symmetries at the end")
    ap.add_argument("--smoke", action="store_true", help="tiny CPU run to check the pipeline end to end")
    ap.add_argument("--predict-only", action="store_true", help="skip training; needs --weights and --postprocess")
    ap.add_argument("--weights", default=None, help="state_dict saved by save_weights (with .json sidecar)")
    ap.add_argument("--postprocess", default=None, help="postprocess.json written by a previous run")
    a = ap.parse_args()
    if a.smoke:
        a.size, a.epochs, a.bs, a.no_pretrained = 256, 1, 2, True
    if a.predict_only and not (a.weights and a.postprocess):
        ap.error("--predict-only needs --weights and --postprocess")
    return a


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def environment_report() -> dict:
    """Versions actually used (copied into requirements.txt after the final Kaggle run)."""
    from importlib.metadata import PackageNotFoundError, version
    rep = {"python": platform.python_version(),
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}
    for pkg in ("torch", "torchvision", "numpy", "opencv-python", "pycocotools",
                "segmentation-models-pytorch", "timm", "pandas"):
        try:
            rep[pkg] = version(pkg)
        except PackageNotFoundError:
            rep[pkg] = "not installed"
    return rep


class Validator:
    """Caches the validation images and GT crops once; scores a model under given post-processing.

    GT is scored per annotator set (170 sets for 106 files), which is exactly how the host's
    self-evaluation notebook groups the ground truth.
    """

    def __init__(self, val_samples, img_dir: Path, size: int, device: str):
        self.size, self.device = size, device
        self.files = sorted({s.file_name for s in val_samples})
        self.imgs = {f: read_gray(img_dir / f) for f in self.files}
        self.gt = [(s.file_name, gt_crops(instance_masks(s))) for s in val_samples]

    def probs(self, model, tta_full: bool = False) -> dict:
        model.eval()
        return {f: predict_prob(model, self.imgs[f], self.size, self.device, tta=True, tta_full=tta_full)
                for f in self.files}

    def score(self, probs: dict, p: PostprocessParams) -> dict:
        labels = {f: prob_to_labels(probs[f], p.thr, p.min_area, p.link_px) for f in self.files}
        return pq_from_stats([pq_stats_labels(gt, labels[f]) for f, gt in self.gt])

    def best_threshold_score(self, probs: dict) -> dict:
        """Rank checkpoints by the best of a few thresholds, not a fixed 0.5 (soft targets
        shift the calibration), with the default min_area / link_px."""
        best = None
        for thr in SELECT_THRS:
            r = self.score(probs, PostprocessParams(thr=thr))
            if best is None or r["PQ"] > best["PQ"]:
                best = {**r, "thr": thr}
        return best


def train_model(a, train_s, val: Validator, img_dir: Path, out: Path, device: str) -> tuple[torch.nn.Module, list]:
    t0 = time.time()
    ds = FilamentDataset(train_s, img_dir, a.size, augment=True)
    print(f"cached {len(ds)} train files in {time.time() - t0:.0f}s", flush=True)
    dl = DataLoader(ds, batch_size=a.bs, shuffle=True, num_workers=2 if device == "cuda" else 0,
                    drop_last=True, pin_memory=device == "cuda",
                    worker_init_fn=lambda w: np.random.seed((torch.initial_seed() + w) % 2**32))

    model = build_model(a.encoder, pretrained=not a.no_pretrained).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=a.epochs * len(dl), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=device == "cuda")
    loss_fn = BceDiceLoss()

    best_pq, log = -1.0, []
    for ep in range(a.epochs):
        model.train()
        t0, tot = time.time(), torch.zeros((), device=device)
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=device == "cuda"):
                loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.detach()
        msg = {"epoch": ep + 1, "loss": round(tot.item() / max(len(dl), 1), 4), "sec": round(time.time() - t0)}
        last = ep == a.epochs - 1
        if a.smoke or (ep + 1) % 5 == 0 or last or (ep >= a.epochs - 10 and (ep + 1) % 2 == 0):
            r = val.best_threshold_score(val.probs(model))
            msg.update({k: round(v, 4) if isinstance(v, float) else v for k, v in r.items()})
            if r["PQ"] > best_pq:
                best_pq = r["PQ"]
                save_weights(model, out / "best.pt", a.encoder, a.size)
        log.append(msg)
        print(msg, flush=True)
    save_weights(model, out / "last.pt", a.encoder, a.size)
    model, _ = load_weights(out / "best.pt", device)
    return model, log


def tune_postprocess(val: Validator, probs: dict) -> list[dict]:
    """Grid search over the post-processing knobs on the cached validation probabilities."""
    results = []
    for thr, ma, lp in itertools.product(DEFAULT_GRID["thr"], DEFAULT_GRID["min_area"], DEFAULT_GRID["link_px"]):
        p = PostprocessParams(thr, ma, lp)
        results.append({**p.to_dict(), **val.score(probs, p)})
    results.sort(key=lambda r: -r["PQ"])
    return results


def predict_test(model, params: PostprocessParams, test_dir: Path, size: int, device: str,
                 tta_full: bool, limit: int | None = None) -> tuple[list, list, list]:
    files = sorted(p.name for p in test_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg"))[:limit]
    rows, empty = [], []
    for f in files:
        prob = predict_prob(model, read_gray(test_dir / f), size, device, tta=True, tta_full=tta_full)
        labels = prob_to_labels(prob, params.thr, params.min_area, params.link_px)
        if labels.max() == 0:
            empty.append(f)
        rows.extend(rows_for_image(Path(f).stem, labels_to_instances(labels)))   # encoded one at a time
    return files, rows, empty


def main() -> None:
    a = parse_args()
    seed_everything(a.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    data_dir = Path(a.data) if a.data else (Path("/kaggle/input") if Path("/kaggle/input").exists()
                                             else Path(__file__).parent / "data")
    root = find_dataset_root(data_dir)
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    env = environment_report()
    print(f"device={device} root={root}\nenv={env}", flush=True)
    with (out / "environment.json").open("w", encoding="utf-8") as fh:
        json.dump(env, fh, indent=1)

    samples = load_samples(root)
    train_s, val_s = split_by_file(samples, a.val_frac)
    if a.smoke:
        train_s, val_s = train_s[:6], val_s[:3]
    print(f"train sets={len(train_s)} ({len({s.file_name for s in train_s})} files)  "
          f"val sets={len(val_s)} ({len({s.file_name for s in val_s})} files)", flush=True)
    val = Validator(val_s, train_image_dir(root), a.size, device)

    if a.predict_only:
        model, meta = load_weights(Path(a.weights), device)
        a.size = meta["size"]
        val.size = a.size
        with open(a.postprocess, encoding="utf-8") as fh:
            params = PostprocessParams.from_dict(json.load(fh))
        probs = val.probs(model, tta_full=not a.no_tta8)
        r = val.score(probs, params)
        print("val PQ with the saved weights/params:", {k: round(v, 4) if isinstance(v, float) else v
                                                        for k, v in r.items()}, flush=True)
        log, grid = [], []
    else:
        model, log = train_model(a, train_s, val, train_image_dir(root), out, device)
        probs = val.probs(model, tta_full=not a.no_tta8)
        grid = tune_postprocess(val, probs)
        params = PostprocessParams.from_dict(grid[0])
        print("top post-processing settings (val PQ):", flush=True)
        for r in grid[:5]:
            print({k: round(v, 4) if isinstance(v, float) else v for k, v in r.items()}, flush=True)
        with (out / "postprocess.json").open("w", encoding="utf-8") as fh:
            json.dump(params.to_dict(), fh)
        with (out / "report.json").open("w", encoding="utf-8") as fh:
            json.dump({"args": vars(a), "environment": env, "train_log": log, "postprocess": params.to_dict(),
                       "val_PQ": grid[0]["PQ"], "postprocess_grid": grid}, fh, indent=1)

    files, rows, empty = predict_test(model, params, test_image_dir(root), a.size, device,
                                      tta_full=not a.no_tta8, limit=3 if a.smoke else None)
    write_submission(rows, out / "submission.csv")
    verify_submission(out / "submission.csv", rows, {Path(f).stem for f in files})
    print(f"submission: {len(rows)} filaments over {len(files)} test images; "
          f"images with no prediction: {len(empty)} {empty[:5]}  -> {out / 'submission.csv'}", flush=True)


if __name__ == "__main__":
    main()


## Run
`-u` makes the progress lines appear as they are printed (the first ~5 min are spent caching the training set, then one line per epoch; validation PQ every 5 epochs and every 2 in the last 10).

In [ ]:
if MODE == 'train':
    !python -u train.py --out /kaggle/working/outputs
else:
    !python -u train.py --predict-only --weights {WEIGHTS_DIR}/best.pt --postprocess {WEIGHTS_DIR}/postprocess.json --out /kaggle/working/outputs

## Check the result
Success looks like: 180 test images, few or no images without a prediction, validation PQ clearly above 0 (the two-annotator agreement reference is 0.35).

In [ ]:
import pandas as pd, json
sub = pd.read_csv('/kaggle/working/outputs/submission.csv')
n_img = sub.filament_id.str.split('_', n=1).str[0].nunique()
print(f'rows={len(sub)}  images with >=1 filament={n_img}/180  (empty={180 - n_img})')
env = json.load(open('/kaggle/working/outputs/environment.json'))
print('environment:', env)
if MODE == 'train':
    rep = json.load(open('/kaggle/working/outputs/report.json'))
    print('chosen post-processing:', rep['postprocess'], ' val PQ:', round(rep['val_PQ'], 4))
    display(pd.DataFrame(rep['train_log']))
    display(pd.DataFrame(rep['postprocess_grid']).head(10))

## Reproducing without training
After a training run, download `outputs/best.pt`, `outputs/best.pt.json` and `outputs/postprocess.json`, upload them as a private Kaggle dataset (e.g. `segmenthreads-weights`), attach it to this notebook, set `MODE = 'predict'` in the first cell and run all. The same files are committed to the public repository under `weights/` for the final deliverable.